#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# set confounders
confounders = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11',
       'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21',
       'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31',
       'x32', 'x33', 'x34', 'x35', 'x36', 'x37', 'x38', 'x39', 'x40', 'x41',
       'x42', 'x43', 'x44', 'x45', 'x46', 'x47', 'x48', 'x49', 'x50', 'x51',
       'x52', 'x53', 'x54', 'x55', 'x56', 'x57', 'x58', 'x59', 'x60', 'x61',
       'x62', 'x63', 'x64', 'x65', 'x66', 'x67', 'x68', 'x69', 'x70', 'x71',
       'x72', 'x73', 'x74', 'x75', 'x76', 'x77', 'x78', 'x79']
input_dim = len(confounders)
scen_list = [4, 32, 45, 47, 60, 62, 75, 77]

#### helpers

In [ ]:
def load_checkpoint(model_cls, path, input_dim, device, hidden_dims=(128, 64)):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {path}")

    last_error = None

    for hidden_dim in hidden_dims:
        try:
            model = model_cls(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
            model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
            model.eval()
            return model
        except RuntimeError as err:
            last_error = err

    raise RuntimeError(f"Could not load checkpoint with hidden dims {hidden_dims}: {path}") from last_error

In [ ]:
def load_models_acic(seed, confounders, device):
    input_dim = len(confounders)

    ranker_dir = ROOT / "experiments" / "supplementary" / "benchmarks" / "acic" /"chkpts" / "rankers" / f"seed_{seed}"
    pointwise_dir = ROOT / "experiments" / "supplementary" / "benchmarks" / "acic" /"chkpts" / "pointwise" / f"seed_{seed}"
    nuisance_dir = ROOT / "experiments" / "supplementary" / "benchmarks" / "acic" /"chkpts" / "nuisances" / f"seed_{seed}"

    orth_model = load_checkpoint(
        ClassificationHead,
        ranker_dir / "orthogonal.pt",
        input_dim,
        device,
    )

    pi_model = load_checkpoint(
        ClassificationHead,
        ranker_dir / "plug_in.pt",
        input_dim,
        device,
    )

    dr_model = load_checkpoint(
        RegressionHead,
        pointwise_dir / "cate_model.pt",
        input_dim,
        device,
    )

    m0_model = load_checkpoint(
        RegressionHead,
        nuisance_dir / "mu0_model.pt",
        input_dim,
        device,
        hidden_dims=(64,),
    )

    m1_model = load_checkpoint(
        RegressionHead,
        nuisance_dir / "mu1_model.pt",
        input_dim,
        device,
        hidden_dims=(64,),
    )

    return orth_model, pi_model, dr_model, m0_model, m1_model

In [ ]:
def pairwise_comparison(df_long, metric="autoc", model_a="orth", model_b="pi", seed_col="seed"):
    wide = df_long.pivot(index=seed_col, columns="model", values=metric)
    diff = wide[model_a] - wide[model_b]

    return pd.Series({
        "metric": metric,
        "model_a": model_a,
        "model_b": model_b,
        "win_rate_a": float((diff > 0).mean())})

#### evaluation

In [ ]:
# init collector
all_metrics = []

# loop over seeds
for seed in scen_list:

    # track progress
    print(f" -> Seed {seed}")
    set_seed(seed)

    # get testing data
    test_df = pd.read_csv(f'./data/datasets/replication_{seed}/test.csv')
    test_df = test_df.rename({'t':'T', 'yf': 'Y', 'tau': 'cate', 'mu0':'M0', 'mu1':'M1'}, axis=1)
    test_loader = DataLoader(EvalDataset(test_df, confounders), batch_size=1024, shuffle=False)

    # load models
    rank_learner, pi_model, dr_model, m0_model, m1_model = load_models_acic(seed=seed, confounders=confounders, device=device)

    # get predictions
    df_eval = get_estimates_all(rank_learner, pi_model, dr_model, m0_model, m1_model, test_loader, device)
    
    # compute and store metrics
    df_metrics = compute_metrics_all(df_eval)
    df_metrics["seed"] = seed
    all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
df_all = df_all.drop(df_all.loc[df_all['model']=='oracle'].index)

In [ ]:
# relative metrics
group = df_all.groupby("seed")["autoc"]
min_per_seed = group.transform("min")
max_per_seed = group.transform("max")

df_all["pct_of_best"] = (df_all["autoc"] - min_per_seed) / (max_per_seed - min_per_seed + 1e-8)
df_all["is_best"] = df_all.groupby("seed")["autoc"].transform("max") == df_all["autoc"]

# summary table
summary = (df_all.groupby("model").agg(relative_autoc=("pct_of_best", "mean"), overall_win_rate=("is_best", "mean")).reset_index())

In [ ]:
# compute win rates
others = ["DR", "T", "plug_in"]
rank_vs_all = pd.concat([pairwise_comparison(df_all, metric="autoc", model_a="rank_learner", model_b=m, seed_col="seed") for m in others], axis=1).T
summary = summary.merge(rank_vs_all[["model_b", "win_rate_a"]].rename(columns={"model_b": "model", "win_rate_a": "rank_learner_win_rate"}), on="model", how="left")
summary.loc[summary["model"] == "rank_learner", "rank_learner_win_rate"] = 0.5